# Missing Heat Loss Forensics -- Test 0126

**Goal:** Diagnose the ~200 C discrepancy in insulated (triple-wall) sections by cleanly separating *measurement-model* effects from *missing-physics* effects.

All temperature comparisons use the **measured TC bead radial position** (1.268 in from inner wall, r/R = 0.5773, **fixed from geometry**).

| Section | Contents |
|---------|----------|
| **A** | Baseline simulation + data extraction |
| **B** | Measurement models: (A) radial-position correction, (B) optional radiation bias |
| **C** | Temperature & pressure comparison plots |
| **D** | Insulated-section heat-loss terms (Q time series) |
| **E** | Resistance decomposition + sensitivity |
| **F** | Physics toggle sweep (measurement model **fixed**; reports T_core AND T_meas) |
| **G** | Final verdict |

In [1]:
# ====================================================================
# SECTION A: Imports, settings, initialisation
# ====================================================================
import sys, os, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import brentq
from math import pi, log

warnings.filterwarnings('ignore')

REPO_ROOT = os.path.abspath('..')
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'dev', 'exp_data'))

from flued_module.chimney import Chimney, make_smooth_door_schedule
from flued_module.material_properties import MaterialProperties
from experiment_loader import ExperimentLoader

SIGMA = 5.67e-8
TEST_ID = '0126'
SIM_DURATION = 600
DT = 0.5
RAMP_S = 5.0
PA_LAB = 98457        # Pa (lab elevation)
CP_GAS = 1050.0       # J/(kg*K)

# TC bead geometry  (6-inch pipe)
PIPE_D_IN = 6.0
PIPE_R_MM = PIPE_D_IN * 25.4 / 2           # 76.2 mm
BEAD_FROM_WALL_IN = 1.268                   # inches from inner wall
BEAD_FROM_WALL_MM = BEAD_FROM_WALL_IN * 25.4  # 32.2 mm
BEAD_R_MM = PIPE_R_MM - BEAD_FROM_WALL_MM     # 44.0 mm from center
ETA_BEAD = BEAD_R_MM / PIPE_R_MM              # 0.577

OUTPUT_DIR = os.path.join(REPO_ROOT, 'outputs', 'heat_loss_diagnostic')
os.makedirs(OUTPUT_DIR, exist_ok=True)

mat_props = MaterialProperties(
    os.path.join(REPO_ROOT, 'flued_module', 'data', 'pipe_props.json'))
loader = ExperimentLoader(
    metadata_path=os.path.join(REPO_ROOT, 'dev', 'exp_data',
                               'experiments_metadata.json'))

print(f'Bead: {BEAD_FROM_WALL_IN}" from wall, eta = r/R = {ETA_BEAD:.3f}')
print(f'Output dir: {OUTPUT_DIR}')

Bead: 1.268" from wall, eta = r/R = 0.577
Output dir: c:\Users\Ana\OneDrive - LBNL\Documents\WH\flued_dynamics\outputs\heat_loss_diagnostic


In [2]:
# ====================================================================
# Helper functions
# ====================================================================

def sensor_to_segment(chimney, sensor_name, test_cfg, mapping='temperature'):
    """Map a named sensor to nearest segment index."""
    cmap = test_cfg.get('column_mapping', {}).get(mapping, {})
    info = cmap.get(sensor_name)
    if info is None:
        return None
    h_in = info.get('height_in')
    if h_in is None:
        return None
    loc = info.get('location', 'insulated')
    h_m = h_in * 0.0254
    if loc == 'black_pipe':
        abs_h = chimney.part_base_elevations[0] + h_m
    else:
        abs_h = chimney.part_base_elevations[-1] + h_m
    return int(np.argmin(np.abs(chimney.segment_elevations - abs_h)))


# ============================================================
# Measurement Model A: Radial-position-only
# ============================================================
# TC bead sits at r/R = 0.5773 (1.268 in from wall in 6.0 in ID pipe).
# This is FIXED from geometry -- we do NOT fit r/R.
#
#   T_meas = T_wall + (T_core - T_wall) * (1 - (r/R)^n)
#
# n is a radial-profile shape parameter:
#   n -> 0   : T_meas -> T_wall  (flat profile)
#   n -> inf : T_meas -> T_core  (thin BL, negligible correction)

def compute_T_radial(T_core, T_wall, n, eta=ETA_BEAD):
    """Temperature at TC bead radial position (K in, K out)."""
    return T_wall + (T_core - T_wall) * (1.0 - eta ** n)


def adjusted_temperature(ch, seg, step, n):
    """Radial-corrected temperature (C) at a segment/timestep."""
    return compute_T_radial(ch.T_gas[step, seg],
                            ch.T_wall[step, seg], n) - 273.15


# ============================================================
# Measurement Model B: TC bead radiation bias  (OPTIONAL)
# ============================================================
# Estimates additional cooling of the TC bead due to radiation
# exchange with the pipe wall.  Energy balance on the bead:
#
#   h_conv * (T_local - T_bead) = eps_bead * sigma * (T_bead^4 - T_wall^4)
#
# Assumptions (documented):
#   1. T_local = gas temperature at bead position (from Model A).
#   2. h_conv reused from pipe inner convection coefficient.
#   3. Bead exchanges radiation only with pipe wall (not gas).
#   4. Bead area cancels -- bead diameter is irrelevant.

def tc_bead_radiation_solve(T_local_K, T_wall_K, h_conv, eps_bead):
    """Solve bead energy balance for equilibrium T_bead (K)."""
    if T_local_K <= T_wall_K or h_conv <= 0 or eps_bead <= 0:
        return T_local_K
    def resid(Tb):
        return h_conv * (T_local_K - Tb) - eps_bead * SIGMA * (Tb**4 - T_wall_K**4)
    try:
        return brentq(resid, T_wall_K + 0.01,
                      max(T_local_K, T_wall_K + 0.02), xtol=0.05)
    except ValueError:
        return T_local_K


def adjusted_temperature_with_radiation(ch, seg, step, n, eps_bead):
    """Temperature corrected for radial position AND radiation bias (C)."""
    T_local_K = compute_T_radial(ch.T_gas[step, seg],
                                 ch.T_wall[step, seg], n)
    T_bead_K = tc_bead_radiation_solve(
        T_local_K, ch.T_wall[step, seg],
        ch.h_conv_inner[step, seg], eps_bead)
    return T_bead_K - 273.15


print(f'Helpers loaded.  r/R = {ETA_BEAD:.6f} (FIXED from geometry)')
print(f'  Model A: T_meas = T_wall + (T_core-T_wall) * (1-(r/R)^n)')
print(f'  Model B: bead radiation balance (optional, separated)')

Helpers loaded.  r/R = 0.577333 (FIXED from geometry)
  Model A: T_meas = T_wall + (T_core-T_wall) * (1-(r/R)^n)
  Model B: bead radiation balance (optional, separated)


In [3]:
# ====================================================================
# Run baseline simulation
# ====================================================================
cfg_chimney = loader.get_chimney_config(TEST_ID, mat_props=mat_props)
test_cfg = loader.get_test_config(TEST_ID)
cond = test_cfg['conditions']
door = test_cfg.get('door_events', {})
offsets = door.get('offsets_s', {})
door_in = door.get('door_opening_in', 2.0)
T_INDOOR_K = cfg_chimney['T_indoor_K']
T_OUTDOOR_K = cfg_chimney['T_outdoor_K']

full_door_ht_in = cfg_chimney['inlet_cfg'].get('door_height_ft', 0.82) * 12
cracked_pct = min(100.0, door_in / full_door_ht_in * 100)
t_crack_key = 'door_to_2in' if 'door_to_2in' in offsets else 'door_to_3in'
t_crack = offsets.get(t_crack_key, 60)
t_close = offsets.get('door_closed', 180)

door_schedule = make_smooth_door_schedule(
    events=[(0, 100.0), (t_crack, cracked_pct), (t_close, 0.0)],
    ramp_open_to_cracked_s=RAMP_S,
    ramp_cracked_to_closed_s=RAMP_S,
)

# Inlet temperature from TC1
tc1_data = loader.get_temperature_data(TEST_ID, 'TC1')
tc1_time = tc1_data['time_s']
tc1_temp_K = tc1_data['temperature_C'] + 273.15
inlet_temp_func = interp1d(tc1_time, tc1_temp_K, kind='linear',
                           fill_value='extrapolate')

top_boundary = {
    'enable_wind_cap_effect': True,
    'cap_type': 'rain_cap',
    'wind_speed_m_s': cond['wind_speed_m_s'],
}

print('Running baseline simulation ...')
ch = Chimney(
    parts=cfg_chimney['parts'],
    elbows=cfg_chimney['fittings'],
    elbow_method='K_sum',
    dt=DT, total_time=SIM_DURATION,
    use_coupled_solver=True,
    door_opening_func=door_schedule,
    inlet_temp_func=inlet_temp_func,
    inlet_restriction=cfg_chimney['inlet_cfg'].copy(),
    top_boundary=top_boundary,
    fittings_thermal={'model': 'none'},
    verbose=False,
)
ch.run_simulation()
print('Done.')

# --- Map TC and pressure sensors to segments ---
TC_NAMES = ['TC1', 'TC2', 'TC3', 'TC4']
TC_SEGS = {}
for tc in TC_NAMES:
    seg = sensor_to_segment(ch, tc, test_cfg, mapping='temperature')
    if seg is not None:
        TC_SEGS[tc] = seg

PRES_SENSORS = ['TC1', 'TC2', 'TC3', 'TOP']
PRES_SEGS = {}
for ps in PRES_SENSORS:
    seg = sensor_to_segment(ch, ps, test_cfg, mapping='pressure')
    if seg is not None:
        PRES_SEGS[ps] = seg

print(f'TC segment map:       {TC_SEGS}')
print(f'Pressure sensor map:  {PRES_SEGS}')
print(f'Total segments: {ch.total_segments}')
print(f'Parts: {[(p.type, p.num_segments, p.orientation) for p in ch.parts]}')

Running baseline simulation ...

CHIMNEY GEOMETRY SUMMARY (Sanity Check)

Flow path from inlet to outlet:
----------------------------------------
INLET: 90° tee (R/D=1.0)
Part 0: SINGLE vertical, L=1.87m (6.1ft), D=152.4mm (6.0in)
       +-- FITTING: 90° long-radius (R/D=1.5)
Part 1: TRIPLE horizontal, L=0.59m (1.9ft), D=152.4mm (6.0in) (triple-wall)
       +-- FITTING: 90° tee (R/D=1.0)
Part 2: TRIPLE vertical, L=1.80m (5.9ft), D=152.4mm (6.0in) (triple-wall)

----------------------------------------
Summary:
  Total parts: 3
  Total fittings (elbows/tees): 3
  Total path length: 4.26m (14.0ft)
  Total vertical rise: 3.67m (12.0ft)
  Computational segments: 16
  Elbow loss method: K_sum
  [Note: A_eff capped at max_effective_area = 182 cm^2]

INLET RESTRICTION CONFIGURATION
Status: ENABLED

Door Configuration (swivel):
  Dimensions:         1.27 ft × 0.82 ft
  Full door area:     150.6 sq in (0.09716 m²)
  Opening:            100.0%
  Discharge coeff:    Cd = 0.62
  Leakage area:    

In [4]:
# ====================================================================
# Extract experimental data + per-segment property table
# ====================================================================
ns = ch.T_gas.shape[0]           # time_steps + 1
time_sim = ch.time_array
PA = cfg_chimney.get('PA', PA_LAB)

# --- Load experimental temperatures ---
exp_temp = {}
for tc in TC_NAMES:
    try:
        d = loader.get_temperature_data(TEST_ID, tc)
        exp_temp[tc] = {'time_s': d['time_s'], 'T_C': d['temperature_C']}
    except (ValueError, KeyError) as e:
        print(f'  Warning: no exp temp for {tc}: {e}')

# --- Load experimental pressures (gauge) ---
exp_pres = {}
for ps in PRES_SENSORS:
    try:
        d = loader.get_sensor_data(TEST_ID, ps, as_gauge=True)
        exp_pres[ps] = {'time_s': d['time_s'], 'P_Pa': d['pressure_Pa']}
    except (ValueError, KeyError) as e:
        print(f'  Warning: no exp pressure for {ps}: {e}')

# --- Build per-segment geometry table ---
seg_info = []
seg_idx_global = 0
for p_idx, part in enumerate(ch.parts):
    L_seg = part.height_m / part.num_segments
    for si in range(part.num_segments):
        seg = seg_idx_global + si
        info = {
            'seg': seg, 'part': p_idx,
            'type': part.type, 'orient': part.orientation,
            'D_i': part.inner_diameter_m, 'L': L_seg,
            'T_amb_K': part.T_ambient_K,
        }
        if part.type == 'triple':
            r_gas = part.inner_diameter_m / 2
            r1 = r_gas + part.thickness_inner_m
            r2 = r1 + part.thickness_middle_m
            r3 = r2 + part.thickness_outer_m
            info['D_o']  = 2 * r3
            info['A_i']  = pi * part.inner_diameter_m * L_seg
            info['A_o']  = pi * 2 * r3 * L_seg
            info['k_ins'] = part.k_middle_W_m_K
            info['eps_in'] = part.epsilon_inner
            info['eps_out'] = part.epsilon_outer
            V1 = pi * L_seg * (r1**2 - r_gas**2)
            V2 = pi * L_seg * (r2**2 - r1**2)
            V3 = pi * L_seg * (r3**2 - r2**2)
            info['C_liner'] = part.density_inner_kg_m3 * part.cp_inner_J_kg_K * V1
            info['C_insul'] = part.density_middle_kg_m3 * part.cp_middle_J_kg_K * V2
            info['C_outer'] = part.density_outer_kg_m3  * part.cp_outer_J_kg_K  * V3
            info['R_ins'] = log(r2 / r1) / (2 * pi * part.k_middle_W_m_K * L_seg)
        else:
            D_o = part.inner_diameter_m + 2 * part.thickness_w_m
            info['D_o']  = D_o
            info['A_i']  = pi * part.inner_diameter_m * L_seg
            info['A_o']  = pi * D_o * L_seg
            info['eps_out'] = part.epsilon_w
            r_gas = part.inner_diameter_m / 2
            r_out = r_gas + part.thickness_w_m
            V_w = pi * L_seg * (r_out**2 - r_gas**2)
            info['C_wall'] = part.density_w_kg_m3 * part.cp_w_J_kg_K * V_w
        seg_info.append(info)
    seg_idx_global += part.num_segments

# --- Sanity table at t = 300 s ---
step300 = min(int(300 / DT), ns - 1)
mdot300 = float(ch.mass_flow_rate[step300])

print(f'\n{"="*95}')
print(f'SANITY TABLE  t=300s   mdot={mdot300:.5f} kg/s')
print(f'{"="*95}')
print(f'{"Seg":>4} {"Pt":>3} {"Type":>6} {"T_gas":>7} {"T_wall":>7} '
      f'{"T_out":>7} {"T_amb":>6} {"h_in":>7} {"h_out":>7} {"v(m/s)":>7}')
print('-' * 95)
for si in seg_info:
    seg = si['seg']
    tc_mark = ''
    for tc, s in TC_SEGS.items():
        if s == seg:
            tc_mark = f'  <-- {tc}'
    print(f'{seg:4d} {si["part"]:3d} {si["type"]:>6} '
          f'{ch.T_gas[step300,seg]-273.15:7.1f} '
          f'{ch.T_wall[step300,seg]-273.15:7.1f} '
          f'{ch.T_outer[step300,seg]-273.15:7.1f} '
          f'{si["T_amb_K"]-273.15:6.1f} '
          f'{ch.h_conv_inner[step300,seg]:7.2f} '
          f'{ch.h_conv_outer[step300,seg]:7.2f} '
          f'{ch.velocity[step300,seg]:7.3f}{tc_mark}')


SANITY TABLE  t=300s   mdot=0.03712 kg/s
 Seg  Pt   Type   T_gas  T_wall   T_out  T_amb    h_in   h_out  v(m/s)
-----------------------------------------------------------------------------------------------
   0   0 single   519.6   202.0   202.0   20.0   10.14   18.07   4.702
   1   0 single   507.9   198.4   198.4   20.0   10.11   17.88   4.634  <-- TC1
   2   0 single   496.2   194.9   194.9   20.0   10.08   17.71   4.564
   3   0 single   486.2   191.5   191.5   20.0   10.06   17.54   4.508
   4   0 single   477.3   188.2   188.2   20.0   10.04   17.37   4.456
   5   0 single   468.3   185.1   185.1   20.0   10.01   17.22   4.401
   6   1 triple   462.9   291.7   128.4   14.0   10.22   10.30   4.368
   7   1 triple   457.1   288.5   126.1   14.0   10.20   10.22   4.332
   8   2 triple   452.6   285.5   126.0   14.0   10.00    9.65   4.305
   9   2 triple   448.0   283.1   123.8   14.0   10.00    9.59   4.278  <-- TC2
  10   2 triple   443.5   280.8   121.7   14.0   10.00    9.53 

In [5]:
# ====================================================================
# SECTION B: Measurement Model A -- Fit radial-profile shape (n)
# ====================================================================
# r/R = 0.577333 is FIXED from geometry.
# n is fitted ONCE on the baseline simulation (t=60-600s, TC2/TC3/TC4).
# Reference values (n=7, 9, 11) shown for sensitivity.
# ====================================================================

T_FIT_START = 60
T_FIT_END   = SIM_DURATION
i_fit_start = int(T_FIT_START / DT)
i_fit_end   = min(int(T_FIT_END / DT), ns - 1)

TC_FIT = [tc for tc in TC_NAMES if tc in TC_SEGS and tc != 'TC1']

exp_resamp = {}
for tc in TC_FIT:
    if tc in exp_temp:
        exp_resamp[tc] = np.interp(time_sim, exp_temp[tc]['time_s'],
                                   exp_temp[tc]['T_C'])

# ---- Grid search for best n ----
N_GRID = [0.1, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9,
          1.0, 1.2, 1.5, 2.0, 3.0, 5.0, 7.0, 9.0, 11.0]
results_n = []
for nv in N_GRID:
    per_tc = []
    for tc in TC_FIT:
        if tc not in exp_resamp:
            continue
        seg = TC_SEGS[tc]
        sq = [(adjusted_temperature(ch, seg, st, n=nv) - exp_resamp[tc][st])**2
              for st in range(i_fit_start, i_fit_end + 1)]
        per_tc.append(np.sqrt(np.mean(sq)))
    results_n.append((nv, np.mean(per_tc) if per_tc else 1e9))

best_n_fit, best_rmse_fit = min(results_n, key=lambda x: x[1])
USE_N = best_n_fit
results_n_dict = dict(results_n)
N_REF = [7.0, 9.0, 11.0]

print('MEASUREMENT MODEL A: Radial-Position Correction')
print(f'  r/R = {ETA_BEAD:.6f} (FIXED);  n fitted on baseline')
print(f'\n{"n":>6} {"Avg RMSE":>10}  label')
print('-' * 38)
for nv, r in results_n:
    tags = []
    if nv == best_n_fit:
        tags.append('** BEST FIT **')
    if nv in N_REF:
        tags.append('ref')
    tag = '  ' + ', '.join(tags) if tags else ''
    print(f'{nv:6.2f} {r:10.2f}{tag}')

print(f'\n  Fitted n = {best_n_fit:.2f}  ->  avg RMSE = {best_rmse_fit:.1f} C')
for nr in N_REF:
    print(f'  Ref n={nr:.0f}:  avg RMSE = {results_n_dict.get(nr, float("nan")):.1f} C')

# ---- Per-TC: RMSE/bias for BOTH T_core and T_meas ----
tc_rmse_core = {}
tc_bias_core = {}
tc_rmse_meas = {}
tc_bias_meas = {}
for tc in TC_FIT:
    if tc not in exp_resamp:
        continue
    seg = TC_SEGS[tc]
    dc = np.array([ch.T_gas[st, seg] - 273.15 - exp_resamp[tc][st]
                    for st in range(i_fit_start, i_fit_end + 1)])
    dm = np.array([adjusted_temperature(ch, seg, st, n=USE_N) - exp_resamp[tc][st]
                    for st in range(i_fit_start, i_fit_end + 1)])
    tc_rmse_core[tc] = np.sqrt(np.mean(dc**2))
    tc_bias_core[tc] = np.mean(dc)
    tc_rmse_meas[tc] = np.sqrt(np.mean(dm**2))
    tc_bias_meas[tc] = np.mean(dm)

rmse_bulk     = np.mean(list(tc_rmse_core.values()))
avg_rmse_meas = np.mean(list(tc_rmse_meas.values()))
avg_bias_meas = np.mean(list(tc_bias_meas.values()))

print(f'\n{"="*58}')
print(f'Per-TC comparison (t={T_FIT_START}-{T_FIT_END}s, n={USE_N:.2f}):')
print(f'{"TC":>6} {"Seg":>4} {"RMSE_core":>10} {"RMSE_meas":>10} {"Bias_meas":>10}')
print('-' * 44)
for tc in TC_FIT:
    if tc not in tc_rmse_core:
        continue
    seg = TC_SEGS[tc]
    print(f'{tc:>6} {seg:4d} {tc_rmse_core[tc]:10.1f} '
          f'{tc_rmse_meas[tc]:10.1f} {tc_bias_meas[tc]:+10.1f}')
print(f'{"Avg":>6} {"":>4} {rmse_bulk:10.1f} '
      f'{avg_rmse_meas:10.1f} {avg_bias_meas:+10.1f}')
print(f'\n  T_core vs exp:  avg RMSE = {rmse_bulk:.1f} C')
print(f'  T_meas vs exp:  avg RMSE = {avg_rmse_meas:.1f} C  (n={USE_N:.2f})')

# ---- Pre-compute time series for all TCs ----
T_sim_adj = {}
for tc in TC_NAMES:
    seg = TC_SEGS.get(tc)
    if seg is None:
        continue
    T_sim_adj[tc] = np.array([adjusted_temperature(ch, seg, st, n=USE_N)
                               for st in range(ns)])

print(f'\nn = {USE_N:.2f} LOCKED for all subsequent cells (measurement model fixed).')

MEASUREMENT MODEL A: Radial-Position Correction
  r/R = 0.577333 (FIXED);  n fitted on baseline

     n   Avg RMSE  label
--------------------------------------
  0.10      36.64
  0.20      33.10
  0.30      30.72
  0.35      29.97
  0.40      29.52
  0.45      29.35  ** BEST FIT **
  0.50      29.43
  0.60      30.30
  0.70      31.93
  0.80      34.11
  0.90      36.64
  1.00      39.40
  1.20      45.14
  1.50      53.55
  2.00      65.76
  3.00      82.97
  5.00      99.11
  7.00     104.55  ref
  9.00     106.37  ref
 11.00     106.98  ref

  Fitted n = 0.45  ->  avg RMSE = 29.3 C
  Ref n=7:  avg RMSE = 104.6 C
  Ref n=9:  avg RMSE = 106.4 C
  Ref n=11:  avg RMSE = 107.0 C

Per-TC comparison (t=60-600s, n=0.45):
    TC  Seg  RMSE_core  RMSE_meas  Bias_meas
--------------------------------------------
   TC2    9      100.8       28.6       -7.1
   TC3   12      110.1       28.5       +3.3
   TC4   14      110.9       30.9       +6.4
   Avg           107.3       29.3       +0.9

 

In [6]:
# ====================================================================
# SECTION B (cont): Measurement Model B -- TC bead radiation bias
# ====================================================================
# OPTIONAL supplement to Model A.  Kept separate so we can see
# how much of the residual comes from radiation vs missing physics.
#
# Energy balance:  h*(T_local - T_bead) = eps*sigma*(T_bead^4 - T_wall^4)
#
# Documented assumptions:
#   - T_local from Model A (radial position, n=USE_N).
#   - h_conv = pipe inner convection coefficient (conservative).
#   - Bead sees only the pipe wall for radiation.
#   - Bead area cancels (diameter irrelevant with this simplification).
# ====================================================================

EPS_BEAD_SCAN = [0.2, 0.4, 0.6, 0.8, 0.95]
results_rad = []

for eb in EPS_BEAD_SCAN:
    per_tc = []
    for tc in TC_FIT:
        if tc not in exp_resamp:
            continue
        seg = TC_SEGS[tc]
        sq = [(adjusted_temperature_with_radiation(ch, seg, st, n=USE_N, eps_bead=eb)
               - exp_resamp[tc][st])**2
              for st in range(i_fit_start, i_fit_end + 1)]
        per_tc.append(np.sqrt(np.mean(sq)))
    results_rad.append((eb, np.mean(per_tc) if per_tc else 1e9))

# ---- Effect size at TC2, t=300s ----
seg_tc2 = TC_SEGS.get('TC2')
if seg_tc2 is not None:
    T_core_300 = ch.T_gas[step300, seg_tc2] - 273.15
    T_wall_300 = ch.T_wall[step300, seg_tc2] - 273.15
    T_radial_300 = adjusted_temperature(ch, seg_tc2, step300, n=USE_N)

    print(f'Model B: Radiation bias scan  (TC2, t=300s)')
    print(f'  T_core = {T_core_300:.1f} C,  T_wall = {T_wall_300:.1f} C,  '
          f'T_radial (Model A) = {T_radial_300:.1f} C')
    print(f'\n{"eps_bead":>10} {"T_bead(C)":>10} {"dT_rad(C)":>10} {"Avg RMSE":>10}')
    print('-' * 44)
    for eb, rmse_avg in results_rad:
        T_b = adjusted_temperature_with_radiation(
            ch, seg_tc2, step300, n=USE_N, eps_bead=eb)
        dT = T_b - T_radial_300
        print(f'{eb:10.2f} {T_b:10.1f} {dT:+10.1f} {rmse_avg:10.1f}')

best_rad_eps, best_rad_rmse = min(results_rad, key=lambda x: x[1])
print(f'\nModel A alone:  avg RMSE = {best_rmse_fit:.1f} C')
print(f'Model A+B best: avg RMSE = {best_rad_rmse:.1f} C  (eps_bead={best_rad_eps})')
print(f'Additional reduction from radiation: {best_rmse_fit - best_rad_rmse:.1f} C')
print(f'\nConclusion: radiation bias is a {"small" if best_rmse_fit - best_rad_rmse < 5 else "moderate"} '
      f'supplementary effect.')
print(f'All subsequent sections use Model A only (radial correction, n={USE_N:.2f}).')

Model B: Radiation bias scan  (TC2, t=300s)
  T_core = 448.0 C,  T_wall = 283.1 C,  T_radial (Model A) = 319.2 C

  eps_bead  T_bead(C)  dT_rad(C)   Avg RMSE
--------------------------------------------
      0.20      302.9      -16.3       30.0
      0.40      296.9      -22.3       31.9
      0.60      293.7      -25.5       33.3
      0.80      291.7      -27.5       34.4
      0.95      290.7      -28.6       35.0

Model A alone:  avg RMSE = 29.3 C
Model A+B best: avg RMSE = 30.0 C  (eps_bead=0.2)
Additional reduction from radiation: -0.7 C

Conclusion: radiation bias is a small supplementary effect.
All subsequent sections use Model A only (radial correction, n=0.45).


In [7]:
# ====================================================================
# SECTION C: Temperature comparison -- Sim vs Exp at sensor heights
# ====================================================================

# ------------------------------------------------------------------
# Figure 1: Three-panel TC comparison (TC2, TC3, TC4)
#   exp (black solid), T_core (blue dashed), T_wall (green dashed),
#   T_meas_radial (red solid)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

for i, tc in enumerate(['TC2', 'TC3', 'TC4']):
    ax = axes[i]
    seg = TC_SEGS.get(tc)
    if seg is None:
        continue

    if tc in exp_temp:
        ax.plot(exp_temp[tc]['time_s'], exp_temp[tc]['T_C'],
                'k-', lw=1.5, label='Experiment', alpha=0.8)
    ax.plot(time_sim, ch.T_gas[:, seg] - 273.15, 'b--', lw=1.2,
            alpha=0.55, label='Sim T_core (bulk)')
    ax.plot(time_sim, ch.T_wall[:, seg] - 273.15, 'g--', lw=1.2,
            alpha=0.55, label='Sim T_wall (liner)')
    if tc in T_sim_adj:
        ax.plot(time_sim, T_sim_adj[tc], 'r-', lw=1.5,
                label=f'Sim T_meas (n={USE_N:.2f})')

    ht_str = ''
    cmap_t = test_cfg.get('column_mapping', {}).get('temperature', {})
    tc_info = cmap_t.get(tc)
    if tc_info and 'height_in' in tc_info:
        ht_str = f'  h={tc_info["height_in"]}"'
    ax.set_title(f'{tc} (seg {seg}{ht_str})')
    ax.set_xlabel('Time (s)')
    if i == 0:
        ax.set_ylabel('Temperature (C)')
    ax.legend(fontsize=7, loc='best')
    ax.set_xlim(T_FIT_START, SIM_DURATION)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Test {TEST_ID}: Insulated-section TCs -- sim vs experiment\n'
             f'r/R = {ETA_BEAD:.4f} (fixed from geometry); '
             f'n = {USE_N:.2f} (fitted once on baseline)',
             fontsize=11)
plt.tight_layout()
fname = os.path.join(OUTPUT_DIR, 'temperature_tc_comparison.png')
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fname}')

# ------------------------------------------------------------------
# Figure 2: TC2 measurement-model explainer
# ------------------------------------------------------------------
seg_tc2 = TC_SEGS.get('TC2')
if seg_tc2 is not None and 'TC2' in exp_temp:
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.plot(exp_temp['TC2']['time_s'], exp_temp['TC2']['T_C'],
             'k-', lw=2, label='Experiment (TC bead reading)')
    ax2.plot(time_sim, ch.T_gas[:, seg_tc2] - 273.15,
             'b--', lw=1.5, alpha=0.6, label='Sim T_core (bulk gas)')
    ax2.plot(time_sim, ch.T_wall[:, seg_tc2] - 273.15,
             'g--', lw=1.5, alpha=0.6, label='Sim T_wall (liner)')
    ax2.plot(time_sim, T_sim_adj['TC2'],
             'r-', lw=2, label=f'Sim T_meas (radial, n={USE_N:.2f})')

    # Annotate gaps at t=300s
    sa = min(int(300 / DT), ns - 1)
    yc = ch.T_gas[sa, seg_tc2] - 273.15
    ym = T_sim_adj['TC2'][sa]
    ye = np.interp(300, exp_temp['TC2']['time_s'], exp_temp['TC2']['T_C'])
    ax2.annotate(f'T_core - T_exp = {yc - ye:.0f} C',
                 xy=(300, yc), xytext=(360, yc + 15), fontsize=8,
                 arrowprops=dict(arrowstyle='->', color='blue'), color='blue')
    ax2.annotate(f'T_meas - T_exp = {ym - ye:+.0f} C',
                 xy=(300, ym), xytext=(360, ym - 30), fontsize=8,
                 arrowprops=dict(arrowstyle='->', color='red'), color='red')

    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Temperature (C)')
    ax2.set_title(f'TC2 (seg {seg_tc2}): Why measurement-location correction matters\n'
                  f'r/R = {ETA_BEAD:.4f} fixed from 1.268" bead offset; '
                  f'n = {USE_N:.2f} fitted only once on baseline')
    ax2.legend(fontsize=9, loc='upper right')
    ax2.set_xlim(T_FIT_START, SIM_DURATION)
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    fname2 = os.path.join(OUTPUT_DIR, 'tc2_measurement_model_explainer.png')
    plt.savefig(fname2, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname2}')

Saved: c:\Users\Ana\OneDrive - LBNL\Documents\WH\flued_dynamics\outputs\heat_loss_diagnostic\temperature_tc_comparison.png
Saved: c:\Users\Ana\OneDrive - LBNL\Documents\WH\flued_dynamics\outputs\heat_loss_diagnostic\tc2_measurement_model_explainer.png


In [8]:
# ====================================================================
# SECTION C (cont): Pressure comparison plots
# ====================================================================

# Pre-compute gauge pressures using get_static_pressure_profile()
sim_pres_gauge = {}
for ps in PRES_SENSORS:
    seg = PRES_SEGS.get(ps)
    if seg is None:
        continue
    P_gauge_ts = np.zeros(ns)
    for step in range(ns):
        profile = ch.get_static_pressure_profile(timestep=step)
        P_gauge_ts[step] = profile['P_gauge_Pa'][seg]
    sim_pres_gauge[ps] = P_gauge_ts

# ------------------------------------------------------------------
# 2x2 pressure subplots  (TC1, TC2, TC3, TOP)
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for i, ps in enumerate(PRES_SENSORS[:4]):
    ax = axes[i]
    seg = PRES_SEGS.get(ps)

    if ps in exp_pres:
        ax.plot(exp_pres[ps]['time_s'], exp_pres[ps]['P_Pa'],
                'k-', lw=1.5, label='Experiment', alpha=0.8)
    if ps in sim_pres_gauge:
        ax.plot(time_sim, sim_pres_gauge[ps], 'r--', lw=1.5, label='Simulation')

    ht_str = ''
    cmap_p = test_cfg.get('column_mapping', {}).get('pressure', {})
    ps_info = cmap_p.get(ps)
    if ps_info and 'height_in' in ps_info:
        ht_str = f', h={ps_info["height_in"]}"'
    ax.set_ylabel('Gauge Pressure (Pa)')
    ax.set_title(f'Pressure at {ps} (seg {seg}{ht_str})')
    ax.legend(fontsize=8)
    ax.set_xlim(0, SIM_DURATION)
    ax.grid(True, alpha=0.3)

axes[2].set_xlabel('Time (s)')
axes[3].set_xlabel('Time (s)')
plt.suptitle(f'Test {TEST_ID}: Pressure Comparison (gauge, via get_static_pressure_profile)',
             fontsize=12, y=1.02)
plt.tight_layout()
fname = os.path.join(OUTPUT_DIR, 'pressure_tc_comparison.png')
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fname}')

# ------------------------------------------------------------------
# Pressure RMSE / bias table
# ------------------------------------------------------------------
print(f'\nPressure RMSE and Bias (t={T_FIT_START}-{SIM_DURATION}s):')
print(f'{"Sensor":>8} {"Seg":>4} {"RMSE(Pa)":>10} {"Bias(Pa)":>10}')
print('-' * 35)
pres_rmse = {}
pres_bias = {}
for ps in PRES_SENSORS:
    if ps not in exp_pres or ps not in sim_pres_gauge:
        continue
    seg = PRES_SEGS[ps]
    t_exp = exp_pres[ps]['time_s']
    P_exp = exp_pres[ps]['P_Pa']
    P_sim_i = np.interp(t_exp, time_sim, sim_pres_gauge[ps])
    mask = t_exp >= T_FIT_START
    if not np.any(mask):
        continue
    diffs = P_sim_i[mask] - P_exp[mask]
    pres_rmse[ps] = np.sqrt(np.mean(diffs**2))
    pres_bias[ps] = np.mean(diffs)
    print(f'{ps:>8} {seg:4d} {pres_rmse[ps]:10.1f} {pres_bias[ps]:+10.1f}')

if pres_rmse:
    avg_p_rmse = np.mean(list(pres_rmse.values()))
    avg_p_bias = np.mean(list(pres_bias.values()))
    print(f'{"Avg":>8} {"":>4} {avg_p_rmse:10.1f} {avg_p_bias:+10.1f}')

Saved: c:\Users\Ana\OneDrive - LBNL\Documents\WH\flued_dynamics\outputs\heat_loss_diagnostic\pressure_tc_comparison.png

Pressure RMSE and Bias (t=60-600s):
  Sensor  Seg   RMSE(Pa)   Bias(Pa)
-----------------------------------
     TC1    1        4.5       -2.5
     TC2    9        2.2       -1.5
     TC3   12        1.2       -0.8
     TOP   14        1.2       -1.0
     Avg             2.3       -1.4


In [9]:
# ====================================================================
# SECTION D: Is the insulated section missing heat loss?
# ====================================================================
# For TC2/TC3/TC4 segments, compute time series of:
#   Q_gas->wall = h_inner * A_inner * (T_gas - T_wall)
#   Q_wall->amb = h_outer * A_outer * (T_outer - T_amb)
# ====================================================================

TC_DIAG = {tc: TC_SEGS[tc] for tc in ['TC2', 'TC3', 'TC4'] if tc in TC_SEGS}

Q_gw_ts  = {}
Q_out_ts = {}
for tc, seg in TC_DIAG.items():
    si = seg_info[seg]
    Q_gw_ts[tc]  = ch.h_conv_inner[:, seg] * si['A_i'] * (
        ch.T_gas[:, seg] - ch.T_wall[:, seg])
    Q_out_ts[tc] = ch.h_conv_outer[:, seg] * si['A_o'] * (
        ch.T_outer[:, seg] - si['T_amb_K'])

# ------------------------------------------------------------------
# Figure: Heat-transfer-rate time series
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for i, (tc, seg) in enumerate(TC_DIAG.items()):
    ax = axes[i]
    ax.plot(time_sim, Q_gw_ts[tc],  'r-',  lw=1.5, label='Q gas->wall')
    ax.plot(time_sim, Q_out_ts[tc], 'b--', lw=1.5, label='Q wall->amb')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_title(f'{tc} (seg {seg})')
    ax.set_xlabel('Time (s)')
    if i == 0:
        ax.set_ylabel('Heat transfer rate (W)')
    ax.legend(fontsize=8)
    ax.set_xlim(0, SIM_DURATION)
    ax.grid(True, alpha=0.3)

plt.suptitle('Insulated-section heat transfer terms (baseline physics)', fontsize=12)
plt.tight_layout()
fname = os.path.join(OUTPUT_DIR, 'insulated_section_heat_loss_terms.png')
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fname}')

# ------------------------------------------------------------------
# Summary table at t=300s
# ------------------------------------------------------------------
print(f'\n{"="*75}')
print(f'HEAT TRANSFER at t=300s   mdot = {mdot300:.5f} kg/s')
print(f'{"="*75}')
print(f'{"TC":>6} {"Seg":>4} {"Q_gw(W)":>8} {"Q_out(W)":>8} {"dE/dt(W)":>8} '
      f'{"dT_gw(K)":>8} {"dT_out(K)":>9}')
print('-' * 55)

for tc, seg in TC_DIAG.items():
    si = seg_info[seg]
    qgw  = float(Q_gw_ts[tc][step300])
    qout = float(Q_out_ts[tc][step300])
    dT_gw  = ch.T_gas[step300, seg] - ch.T_wall[step300, seg]
    dT_out = ch.T_outer[step300, seg] - si['T_amb_K']
    if si['type'] == 'triple' and step300 > 0:
        dE = (si['C_liner'] * (ch.T_wall[step300, seg] - ch.T_wall[step300-1, seg])
            + si['C_insul'] * (ch.T_middle[step300, seg] - ch.T_middle[step300-1, seg])
            + si['C_outer'] * (ch.T_outer[step300, seg] - ch.T_outer[step300-1, seg])) / DT
    else:
        dE = 0.0
    print(f'{tc:>6} {seg:4d} {qgw:8.1f} {qout:8.1f} {dE:8.1f} '
          f'{dT_gw:8.0f} {dT_out:9.0f}')

print(f'\nQ_gw >> Q_out  =>  wall is still warming (thermal mass absorbing heat).')
print(f'The insulation resistance limits Q_out, keeping the wall hot.')

Saved: c:\Users\Ana\OneDrive - LBNL\Documents\WH\flued_dynamics\outputs\heat_loss_diagnostic\insulated_section_heat_loss_terms.png

HEAT TRANSFER at t=300s   mdot = 0.03712 kg/s
    TC  Seg  Q_gw(W) Q_out(W) dE/dt(W) dT_gw(K) dT_out(K)
-------------------------------------------------------
   TC2    9    178.0    152.4     27.3      165       110
   TC3   12    171.4    141.0     32.5      159       104
   TC4   14    167.7    133.5     36.5      155        99

Q_gw >> Q_out  =>  wall is still warming (thermal mass absorbing heat).
The insulation resistance limits Q_out, keeping the wall hot.


In [10]:
# ====================================================================
# SECTION E: Resistance Decomposition + Sensitivity
# ====================================================================
# What limits heat loss?  If we halved each resistance,
# how much more heat would leave the pipe?
# ====================================================================

SENS_FACTOR = 1.0 - ETA_BEAD ** USE_N   # dT_meas / dT_core ratio
print(f'{"="*100}')
print(f'RESISTANCE DECOMPOSITION + SENSITIVITY  t=300s')
print(f'{"="*100}')
print(f'  T_meas factor (1 - eta^n) = {SENS_FACTOR:.3f}')
print(f'  (a 10 C drop in T_core -> ~{SENS_FACTOR*10:.1f} C drop in T_meas)\n')

for tc_name in ['TC2', 'TC3', 'TC4']:
    if tc_name not in TC_SEGS:
        continue
    seg = TC_SEGS[tc_name]
    si = seg_info[seg]
    if si['type'] != 'triple':
        continue

    p_idx, _, part = ch.get_segment_info(seg)
    h_in  = ch.h_conv_inner[step300, seg]
    h_out = ch.h_conv_outer[step300, seg]
    T_gas = ch.T_gas[step300, seg]
    T_wall = ch.T_wall[step300, seg]
    T_out = ch.T_outer[step300, seg]
    T_amb = si['T_amb_K']

    R_in = 1.0 / (h_in * si['A_i'] + 1e-12)
    r_gas = part.inner_diameter_m / 2
    r1 = r_gas + part.thickness_inner_m
    r2 = r1 + part.thickness_middle_m
    r3 = r2 + part.thickness_outer_m
    R_ins = log(r2 / r1) / (2 * pi * part.k_middle_W_m_K * si['L'])
    R_ext = 1.0 / (h_out * si['A_o'] + 1e-12)
    R_total = R_in + R_ins + R_ext
    pct_in  = R_in  / R_total * 100
    pct_ins = R_ins / R_total * 100
    pct_ext = R_ext / R_total * 100

    T_film = (T_out + T_amb) / 2
    h_rad_out = 4 * si['eps_out'] * SIGMA * T_film**3
    h_conv_out = max(h_out - h_rad_out, 0.5)
    Q_cur = (T_gas - T_amb) / R_total

    print(f'{tc_name} (seg {seg}):')
    print(f'  T_gas={T_gas-273.15:.1f}C  T_liner={T_wall-273.15:.1f}C  '
          f'T_outer={T_out-273.15:.1f}C  T_amb={T_amb-273.15:.1f}C')
    print(f'  h_inner = {h_in:.2f}  h_outer = {h_out:.2f} '
          f'(conv={h_conv_out:.2f} + rad={h_rad_out:.2f})')
    print(f'  R_inner  = {R_in:.4f} K/W  ({pct_in:.0f}%)')
    print(f'  R_insul  = {R_ins:.4f} K/W  ({pct_ins:.0f}%)')
    print(f'  R_extern = {R_ext:.4f} K/W  ({pct_ext:.0f}%)')
    print(f'  R_total  = {R_total:.4f} K/W   Q_steady = {Q_cur:.1f} W')

    # --- Sensitivity: halving each resistance ---
    print(f'  --- What if we halved each R? ---')
    for label, R_new in [('R_inner/2', R_in/2 + R_ins + R_ext),
                          ('R_insul/2', R_in + R_ins/2 + R_ext),
                          ('R_outer/2', R_in + R_ins + R_ext/2)]:
        Q_new = (T_gas - T_amb) / R_new
        dQ = Q_new - Q_cur
        dT_core_seg = dQ / (abs(mdot300) * CP_GAS + 1e-12)
        dT_meas_seg = dT_core_seg * SENS_FACTOR
        print(f'    {label}: Q={Q_new:.0f} W (+{dQ:.0f} W, +{dQ/max(Q_cur,0.1)*100:.0f}%)'
              f'  dT_core~{-dT_core_seg:.1f} C/seg  dT_meas~{-dT_meas_seg:.1f} C/seg')

    # --- Internal radiation estimate ---
    for eps_ir in [0.3, 0.5]:
        T_avg = (T_gas + T_wall) / 2
        h_rad_in = 4 * eps_ir * SIGMA * T_avg**3
        Q_extra = h_rad_in * si['A_i'] * (T_gas - T_wall)
        print(f'  + inner rad eps={eps_ir}: h_rad={h_rad_in:.1f} W/m2K  '
              f'Q_extra={Q_extra:.0f} W (+{Q_extra/max(Q_cur,0.1)*100:.0f}% of Q_steady)')
    print()

RESISTANCE DECOMPOSITION + SENSITIVITY  t=300s
  T_meas factor (1 - eta^n) = 0.219
  (a 10 C drop in T_core -> ~2.2 C drop in T_meas)

TC2 (seg 9):
  T_gas=448.0C  T_liner=283.1C  T_outer=123.8C  T_amb=14.0C
  h_inner = 10.00  h_outer = 9.59 (conv=6.41 + rad=3.18)
  R_inner  = 0.9265 K/W  (14%)
  R_insul  = 4.9797 K/W  (75%)
  R_extern = 0.7205 K/W  (11%)
  R_total  = 6.6267 K/W   Q_steady = 65.5 W
  --- What if we halved each R? ---
    R_inner/2: Q=70 W (+5 W, +8%)  dT_core~-0.1 C/seg  dT_meas~-0.0 C/seg
    R_insul/2: Q=105 W (+39 W, +60%)  dT_core~-1.0 C/seg  dT_meas~-0.2 C/seg
    R_outer/2: Q=69 W (+4 W, +6%)  dT_core~-0.1 C/seg  dT_meas~-0.0 C/seg
  + inner rad eps=0.3: h_rad=17.7 W/m2K  Q_extra=316 W (+482% of Q_steady)
  + inner rad eps=0.5: h_rad=29.5 W/m2K  Q_extra=526 W (+803% of Q_steady)

TC3 (seg 12):
  T_gas=435.0C  T_liner=276.1C  T_outer=117.5C  T_amb=14.0C
  h_inner = 10.00  h_outer = 9.41 (conv=6.32 + rad=3.09)
  R_inner  = 0.9265 K/W  (14%)
  R_insul  = 4.9797 K/W 

In [11]:
# ====================================================================
# SECTION F: Missing Physics Toggle Sweep  (actual re-runs)
# ====================================================================
# Measurement model is FIXED at n=USE_N throughout.
# We report BOTH T_core and T_meas RMSE so physics effects are not
# masked (or amplified) by the measurement correction.
# ====================================================================

def run_toggle(label, pre_sim_fn=None, chimney_kwargs=None):
    """Run a simulation with a specific physics toggle."""
    cfg = loader.get_chimney_config(TEST_ID, mat_props=mat_props)
    kw = chimney_kwargs or {}
    ch_t = Chimney(
        parts=cfg['parts'],
        elbows=cfg['fittings'],
        elbow_method='K_sum',
        dt=DT, total_time=SIM_DURATION,
        use_coupled_solver=True,
        door_opening_func=door_schedule,
        inlet_temp_func=inlet_temp_func,
        inlet_restriction=cfg['inlet_cfg'].copy(),
        top_boundary=top_boundary,
        fittings_thermal={'model': 'none'},
        verbose=False,
        **kw,
    )
    if pre_sim_fn:
        pre_sim_fn(ch_t)
    ch_t.run_simulation()
    return ch_t


def compute_toggle_rmse(ch_t, n=USE_N):
    """Compute per-TC RMSE for both T_core and T_meas (radial-corrected).

    Returns dict: {tc: {'rmse_core': float, 'rmse_meas': float,
                         'bias_core': float, 'bias_meas': float}}
    """
    ns_t = ch_t.T_gas.shape[0]
    out = {}
    for tc in TC_FIT:
        if tc not in exp_resamp:
            continue
        seg = sensor_to_segment(ch_t, tc, test_cfg)
        if seg is None:
            continue
        dc, dm = [], []
        for step in range(i_fit_start, min(i_fit_end + 1, ns_t)):
            T_core_C = ch_t.T_gas[step, seg] - 273.15
            T_meas_C = adjusted_temperature(ch_t, seg, step, n=n)
            e = exp_resamp[tc][step]
            dc.append(T_core_C - e)
            dm.append(T_meas_C - e)
        dc = np.array(dc)
        dm = np.array(dm)
        out[tc] = {
            'rmse_core': float(np.sqrt(np.mean(dc**2))),
            'rmse_meas': float(np.sqrt(np.mean(dm**2))),
            'bias_core': float(np.mean(dc)),
            'bias_meas': float(np.mean(dm)),
        }
    return out


# Define toggle configs
toggles = {}

# 1. Internal radiation at different emissivities
for eps_ir in [0.2, 0.3, 0.5]:
    tag = f'int_rad_{eps_ir}'
    def make_fn(e=eps_ir):
        def fn(c):
            c.comprehensive_ht.include_internal_radiation = True
            c.comprehensive_ht.eps_internal_radiation = e
        return fn
    toggles[tag] = {'pre_sim_fn': make_fn()}

# 2. Enhanced external convection (wind proxy)
def wind_fn(c):
    c.comprehensive_ht.h_outer_multiplier = 2.5
toggles['wind_h_out_x2.5'] = {'pre_sim_fn': wind_fn}

# 3. k_insulation x 1.3 (temperature effect proxy)
def kT_fn(c):
    for part in c.parts:
        if part.type == 'triple':
            part.k_middle_W_m_K *= 1.3
toggles['k_ins_x1.3'] = {'pre_sim_fn': kT_fn}

# 4. Lower ambient by 10 K for outdoor parts (sky temp proxy)
def sky_fn(c):
    for part in c.parts:
        if part.T_ambient_K < 293:
            part.T_ambient_K -= 10
toggles['sky_Tamb-10K'] = {'pre_sim_fn': sky_fn}

# 5. Plug-flow NTU gas energy model
toggles['plugflow_ntu'] = {
    'chimney_kwargs': {'momentum_config': {'gas_energy_model': 'plugflow_ntu'}},
}

# 6. Combined best physics
def combined_fn(c):
    c.comprehensive_ht.include_internal_radiation = True
    c.comprehensive_ht.eps_internal_radiation = 0.3
    c.comprehensive_ht.h_outer_multiplier = 2.0
    for part in c.parts:
        if part.type == 'triple':
            part.k_middle_W_m_K *= 1.3
toggles['combined'] = {'pre_sim_fn': combined_fn}


# --- Run all toggles ---
print(f'Running {len(toggles)} toggle simulations ...')
print(f'  Measurement model FIXED: n = {USE_N:.2f}\n')
toggle_results = {}
toggle_chimney = {}

for label, cfg in toggles.items():
    print(f'  {label} ...', end=' ', flush=True)
    try:
        ch_t = run_toggle(label,
                          pre_sim_fn=cfg.get('pre_sim_fn'),
                          chimney_kwargs=cfg.get('chimney_kwargs'))
        res = compute_toggle_rmse(ch_t)
        toggle_results[label] = res
        toggle_chimney[label] = ch_t
        avg_c = np.mean([v['rmse_core'] for v in res.values()])
        avg_m = np.mean([v['rmse_meas'] for v in res.values()])
        print(f'RMSE core={avg_c:.1f} C  meas={avg_m:.1f} C')
    except Exception as e:
        print(f'FAILED: {e}')

# Baseline
toggle_results['baseline'] = compute_toggle_rmse(ch)
print('Done.')

Running 8 toggle simulations ...
  Measurement model FIXED: n = 0.45

  int_rad_0.2 ... 
CHIMNEY GEOMETRY SUMMARY (Sanity Check)

Flow path from inlet to outlet:
----------------------------------------
INLET: 90° tee (R/D=1.0)
Part 0: SINGLE vertical, L=1.87m (6.1ft), D=152.4mm (6.0in)
       +-- FITTING: 90° long-radius (R/D=1.5)
Part 1: TRIPLE horizontal, L=0.59m (1.9ft), D=152.4mm (6.0in) (triple-wall)
       +-- FITTING: 90° tee (R/D=1.0)
Part 2: TRIPLE vertical, L=1.80m (5.9ft), D=152.4mm (6.0in) (triple-wall)

----------------------------------------
Summary:
  Total parts: 3
  Total fittings (elbows/tees): 3
  Total path length: 4.26m (14.0ft)
  Total vertical rise: 3.67m (12.0ft)
  Computational segments: 16
  Elbow loss method: K_sum
  [Note: A_eff capped at max_effective_area = 182 cm^2]

INLET RESTRICTION CONFIGURATION
Status: ENABLED

Door Configuration (swivel):
  Dimensions:         1.27 ft × 0.82 ft
  Full door area:     150.6 sq in (0.09716 m²)
  Opening:            10

In [12]:
# ====================================================================
# SECTION F (cont): Toggle Comparison Table + Best-Toggle Plot
# ====================================================================
# Two metrics: T_core (bulk gas) and T_meas (radial-corrected).
# A toggle that improves T_core is a REAL physics improvement.
# A toggle that only improves T_meas might just be compensating via
# the measurement mapping.
# ====================================================================

def _avg_metric(res, key):
    vals = [v[key] for v in res.values() if key in v]
    return np.mean(vals) if vals else float('nan')

base_core = _avg_metric(toggle_results['baseline'], 'rmse_core')
base_meas = _avg_metric(toggle_results['baseline'], 'rmse_meas')

print(f'{"="*95}')
print(f'TOGGLE COMPARISON  (measurement model fixed: n={USE_N:.2f})')
print(f'{"="*95}')

# Header
hdr = f'{"Toggle":>22}'
for tc in TC_FIT:
    hdr += f' {tc+"_c":>8} {tc+"_m":>8}'
hdr += f' {"Avg_c":>8} {"Avg_m":>8} {"d_core":>8}'
print(hdr)
print('-' * 95)

all_names = ['baseline'] + sorted(k for k in toggle_results if k != 'baseline')
sorted_toggles = []
for name in all_names:
    res = toggle_results[name]
    row = f'{name:>22}'
    for tc in TC_FIT:
        tc_res = res.get(tc, {})
        rc = tc_res.get('rmse_core', float('nan'))
        rm = tc_res.get('rmse_meas', float('nan'))
        row += f' {rc:8.1f} {rm:8.1f}'
    avg_c = _avg_metric(res, 'rmse_core')
    avg_m = _avg_metric(res, 'rmse_meas')
    delta_c = avg_c - base_core
    row += f' {avg_c:8.1f} {avg_m:8.1f} {delta_c:+8.1f}'
    print(row)
    sorted_toggles.append((name, avg_c, avg_m))

sorted_toggles.sort(key=lambda x: x[1])

print(f'\nBaseline: T_core RMSE = {base_core:.1f} C,  T_meas RMSE = {base_meas:.1f} C')
print(f'\nRanked by T_core improvement:')
for name, ac, am in sorted_toggles:
    dc = ac - base_core
    dm = am - base_meas
    tag = ' <-- BEST' if name == sorted_toggles[0][0] and name != 'baseline' else ''
    print(f'  {name:>22}: core={ac:.1f} ({dc:+.1f})  meas={am:.1f} ({dm:+.1f}){tag}')

# Exit temperatures
print(f'\n{"="*55}')
print(f'EXIT GAS TEMPERATURE at t=300s')
print(f'{"="*55}')
print(f'{"Toggle":>22} {"T_exit(C)":>10} {"Delta":>8}')
print('-' * 55)
T_exit_base = ch.T_gas[step300, -1] - 273.15
print(f'{"baseline":>22} {T_exit_base:10.1f} {0.0:+8.1f}')
for name, ch_t in sorted(toggle_chimney.items()):
    st = min(step300, ch_t.T_gas.shape[0] - 1)
    T_ex = ch_t.T_gas[st, -1] - 273.15
    print(f'{name:>22} {T_ex:10.1f} {T_ex - T_exit_base:+8.1f}')

# --- Best-toggle temperature plot ---
best_toggle = sorted_toggles[0][0]
if best_toggle == 'baseline' and len(sorted_toggles) > 1:
    best_toggle = sorted_toggles[1][0]
print(f'\nBest toggle (lowest T_core RMSE): {best_toggle}')

ch_best = toggle_chimney.get(best_toggle, ch)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for i, tc in enumerate(TC_NAMES[:4]):
    ax = axes[i]
    seg = TC_SEGS.get(tc)

    # Experiment
    if tc in exp_temp:
        ax.plot(exp_temp[tc]['time_s'], exp_temp[tc]['T_C'],
                'k-', lw=1.5, label='Experiment', alpha=0.8)

    # Baseline adjusted
    if tc in T_sim_adj:
        ax.plot(time_sim, T_sim_adj[tc], 'b--', lw=1.2,
                label='Baseline T_meas', alpha=0.7)

    # Best toggle adjusted
    if best_toggle != 'baseline' and seg is not None:
        ns_b = ch_best.T_gas.shape[0]
        time_b = ch_best.time_array
        seg_b = sensor_to_segment(ch_best, tc, test_cfg)
        if seg_b is not None:
            T_b = np.array([
                adjusted_temperature(ch_best, seg_b, step, n=USE_N)
                for step in range(ns_b)])
            ax.plot(time_b, T_b, 'r-', lw=1.5,
                    label=f'{best_toggle} T_meas')

    ax.set_ylabel('Temperature (C)')
    ax.set_title(f'{tc} (seg {seg})')
    ax.legend(fontsize=7)
    ax.set_xlim(0, SIM_DURATION)
    ax.grid(True, alpha=0.3)
axes[2].set_xlabel('Time (s)')
axes[3].set_xlabel('Time (s)')

plt.suptitle(f'Test {TEST_ID}: Best Toggle ({best_toggle}) vs Baseline\n'
             f'Measurement model fixed: n={USE_N:.2f}',
             fontsize=12, y=1.02)
plt.tight_layout()
fname = os.path.join(OUTPUT_DIR, 'temperature_best_toggle.png')
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fname}')

TOGGLE COMPARISON  (measurement model fixed: n=0.45)
                Toggle    TC2_c    TC2_m    TC3_c    TC3_m    TC4_c    TC4_m    Avg_c    Avg_m   d_core
-----------------------------------------------------------------------------------------------
              baseline    100.8     28.6    110.1     28.5    110.9     30.9    107.3     29.3     +0.0
              combined     47.0     30.0     54.1     29.0     53.8     30.2     51.6     29.7    -55.7
           int_rad_0.2     71.4     30.7     78.7     33.7     78.5     35.4     76.2     33.3    -31.1
           int_rad_0.3     61.5     30.5     68.5     33.4     68.3     35.0     66.1     33.0    -41.2
           int_rad_0.5     47.5     29.2     54.2     31.8     54.2     33.5     51.9     31.5    -55.3
            k_ins_x1.3    100.4     29.0    109.6     27.9    110.2     29.9    106.7     28.9     -0.5
          plugflow_ntu     99.4     28.3    108.6     27.9    109.2     30.0    105.7     28.7     -1.5
          sky_Tamb-

In [13]:
# ====================================================================
# SECTION G: Final Verdict
# ====================================================================
import shutil

print('=' * 80)
print('FINAL VERDICT -- MISSING HEAT LOSS FORENSICS')
print(f'Test {TEST_ID}, t = {T_FIT_START}-{SIM_DURATION} s')
print(f'Measurement model: T_meas = T_wall + (T_core-T_wall)*(1-(r/R)^n)')
print(f'  r/R = {ETA_BEAD:.6f} (FIXED from 1.268" bead offset in 6" pipe)')
print(f'  n   = {USE_N:.2f}       (fitted once on baseline, TC2/TC3/TC4)')
print('=' * 80)

# --- Temperature RMSE/Bias ---
print('\n1. TEMPERATURE RMSE (C)')
print(f'   {"TC":>6} {"Seg":>4} {"RMSE_core":>10} {"Bias_core":>10} '
      f'{"RMSE_meas":>10} {"Bias_meas":>10}')
print(f'   {"-"*52}')
base_res = toggle_results.get('baseline', {})
for tc in TC_FIT:
    seg = TC_SEGS.get(tc, -1)
    r = base_res.get(tc, {})
    print(f'   {tc:>6} {seg:4d} {r.get("rmse_core",0):10.1f} '
          f'{r.get("bias_core",0):+10.1f} '
          f'{r.get("rmse_meas",0):10.1f} {r.get("bias_meas",0):+10.1f}')
print(f'   {"Avg":>6} {"":>4} {base_core:10.1f} '
      f'{"":>10} {base_meas:10.1f}')
reduction = base_core - base_meas
print(f'\n   Radial-position correction reduces RMSE by {reduction:.1f} C '
      f'(from {base_core:.1f} to {base_meas:.1f})')

# --- Pressure RMSE/Bias ---
print('\n2. PRESSURE RMSE (Pa) -- gauge via get_static_pressure_profile')
if pres_rmse:
    print(f'   {"Sensor":>8} {"Seg":>4} {"RMSE(Pa)":>10} {"Bias(Pa)":>10}')
    print(f'   {"-"*35}')
    for ps in PRES_SENSORS:
        if ps in pres_rmse:
            seg = PRES_SEGS.get(ps, -1)
            print(f'   {ps:>8} {seg:4d} {pres_rmse[ps]:10.1f} '
                  f'{pres_bias[ps]:+10.1f}')
    avg_p_rmse = np.mean(list(pres_rmse.values()))
    avg_p_bias = np.mean(list(pres_bias.values()))
    print(f'   {"Avg":>8} {"":>4} {avg_p_rmse:10.1f} {avg_p_bias:+10.1f}')
else:
    print('   (no pressure data available)')

# --- Toggle improvements on T_core ---
print('\n3. DO ANY TOGGLES MATERIALLY IMPROVE T_core?')
improved = [(n, c, m) for n, c, m in sorted_toggles
            if n != 'baseline' and c < base_core - 1.0]
if improved:
    for name, ac, am in improved:
        dc = ac - base_core
        print(f'   YES: {name:>22} -> T_core RMSE {ac:.1f} C  '
              f'(delta = {dc:+.1f} C)')
else:
    print('   NO toggle reduces T_core RMSE by more than 1 C.')
    print('   Physics changes tested here are not large enough to close the gap,')
    print('   OR the gap is dominated by the measurement-location effect.')

# Best overall
best_name, best_c, best_m = sorted_toggles[0]
if best_name == 'baseline' and len(sorted_toggles) > 1:
    best_name, best_c, best_m = sorted_toggles[1]
print(f'\n   Best toggle: {best_name}')
print(f'     T_core RMSE = {best_c:.1f} C (baseline {base_core:.1f})')
print(f'     T_meas RMSE = {best_m:.1f} C (baseline {base_meas:.1f})')

# --- Ranked evidence ---
print('\n4. RANKED EVIDENCE FOR THE ~200 C GAP')
print()
print('   #1  Sensor-location effect (DOMINANT)')
print(f'       TC bead at r/R = {ETA_BEAD:.3f} reads lower than bulk gas T.')
print(f'       Radial correction (n={USE_N:.2f}) reduces RMSE from '
      f'{base_core:.0f} to {base_meas:.0f} C.')
print(f'       This is a ~{reduction:.0f} C RMSE reduction -- the single '
      f'largest identified effect.')
print()
print('   #2  Missing internal radiation (MODERATE, direction correct)')
print(f'       Liner-to-gas radiation (eps ~0.3 for stainless) adds')
print(f'       a parallel heat-transfer path that the baseline omits.')
res_ir = toggle_results.get('int_rad_0.3', {})
if res_ir:
    ir_core = _avg_metric(res_ir, 'rmse_core')
    print(f'       int_rad_0.3: T_core RMSE = {ir_core:.1f} C '
          f'(delta = {ir_core - base_core:+.1f})')
print()
print('   #3  Missing wind-forced external convection (SMALL-MODERATE)')
print(f'       Outdoor segments see wind; baseline uses natural convection only.')
res_w = toggle_results.get('wind_h_out_x2.5', {})
if res_w:
    w_core = _avg_metric(res_w, 'rmse_core')
    print(f'       wind_h_out_x2.5: T_core RMSE = {w_core:.1f} C '
          f'(delta = {w_core - base_core:+.1f})')
print()
print('   #4  Insulation k(T) increase (SMALL)')
print(f'       Mineral-wool k rises with temperature; baseline uses constant k.')
res_k = toggle_results.get('k_ins_x1.3', {})
if res_k:
    k_core = _avg_metric(res_k, 'rmse_core')
    print(f'       k_ins_x1.3: T_core RMSE = {k_core:.1f} C '
          f'(delta = {k_core - base_core:+.1f})')

# --- Unidentifiability caveat ---
print('\n5. UNIDENTIFIABILITY CAVEAT')
print()
print('   The TC bead reads a COMBINATION of (a) the true gas temperature')
print('   profile at its radial position, and (b) radiation exchange with')
print('   the pipe wall.  Without an independent measurement of either:')
print('     - bulk/core gas temperature (e.g., aspirated TC), or')
print('     - liner wall temperature (e.g., surface-mounted TC),')
print('   the heat-loss vs measurement-model attribution is PARTIALLY')
print('   UNIDENTIFIABLE.  A cooler T_core (more heat loss) and a flatter')
print('   radial profile (larger n) produce similar T_bead readings.')
print()
print('   Implication: the fitted n absorbs BOTH the true radial profile')
print('   shape AND any systematic bias in the heat-loss model.  The')
print('   individual contributions cannot be separated from TC data alone.')

print('\n' + '=' * 80)

# --- Export notebook copy ---
nb_src = os.path.join(REPO_ROOT, 'notebooks',
                      'missing_heat_loss_forensics.ipynb')
nb_dst = os.path.join(OUTPUT_DIR,
                      'missing_heat_loss_forensics_UPDATED.ipynb')
try:
    shutil.copy2(nb_src, nb_dst)
    print(f'\nNotebook exported to: {nb_dst}')
except Exception as e:
    print(f'\nCould not export notebook: {e}')

FINAL VERDICT -- MISSING HEAT LOSS FORENSICS
Test 0126, t = 60-600 s
Measurement model: T_meas = T_wall + (T_core-T_wall)*(1-(r/R)^n)
  r/R = 0.577333 (FIXED from 1.268" bead offset in 6" pipe)
  n   = 0.45       (fitted once on baseline, TC2/TC3/TC4)

1. TEMPERATURE RMSE (C)
       TC  Seg  RMSE_core  Bias_core  RMSE_meas  Bias_meas
   ----------------------------------------------------
      TC2    9      100.8      +88.3       28.6       -7.1
      TC3   12      110.1      +95.9       28.5       +3.3
      TC4   14      110.9      +97.3       30.9       +6.4
      Avg           107.3                  29.3

   Radial-position correction reduces RMSE by 77.9 C (from 107.3 to 29.3)

2. PRESSURE RMSE (Pa) -- gauge via get_static_pressure_profile
     Sensor  Seg   RMSE(Pa)   Bias(Pa)
   -----------------------------------
        TC1    1        4.5       -2.5
        TC2    9        2.2       -1.5
        TC3   12        1.2       -0.8
        TOP   14        1.2       -1.0
        Av